In [ ]:
# Bitcoin AI Predictor – Data Analytics Analysis
**Author:** Aryan Kaushik  
**Objective:** End-to-end data analytics workflow for Bitcoin price direction analysis


## 1. Imports & Configuration
Libraries required for data analysis, visualization, and modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.rcParams['figure.figsize'] = (10, 5)


## 2. Data Loading
Historical Bitcoin price data is loaded and inspected.


In [ ]:
df = pd.read_csv('data/bitcoin_data.csv')
df.head()


## 3. Data Cleaning
Standardizing column names, parsing dates, and sorting time-series data.


In [ ]:
df.columns = df.columns.str.lower()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

df.isna().sum()


## 4. Feature Engineering
Rolling indicators and volatility-based features are created.
Target variable represents next-day price direction.


In [ ]:
df['return'] = df['close'].pct_change()

df['ma_7'] = df['close'].rolling(7).mean()
df['ma_21'] = df['close'].rolling(21).mean()
df['volatility_7'] = df['return'].rolling(7).std()

df['target'] = (df['return'].shift(-1) > 0).astype(int)

df = df.dropna().reset_index(drop=True)


## 5. Exploratory Data Analysis (EDA)
Visual inspection of price trends, returns distribution, and feature correlations.


In [ ]:
plt.plot(df['date'], df['close'])
plt.title('Bitcoin Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.show()

plt.hist(df['return'], bins=50)
plt.title('Distribution of Daily Returns')
plt.show()

sns.heatmap(df[['return', 'ma_7', 'ma_21', 'volatility_7']].corr(), annot=True)
plt.title('Feature Correlation Matrix')
plt.show()


## 6. Modeling Strategy
A time-based split is used to avoid data leakage.
Logistic Regression is chosen as a transparent baseline model.


In [ ]:
features = ['ma_7', 'ma_21', 'volatility_7']
X = df[features]
y = df['target']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 7. Baseline Accuracy
Majority-class accuracy used as a sanity check.


In [ ]:
baseline_accuracy = max(y_test.mean(), 1 - y_test.mean())
baseline_accuracy


## 8. Logistic Regression Results


In [ ]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy_score(y_test, y_pred)
print(classification_report(y_test, y_pred))


## 9. Model Evaluation
Confusion matrix for error inspection.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()


## 10. Time-Series Cross Validation


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(model, scaler.fit_transform(X), y, cv=tscv)
cv_scores


## 11. Key Takeaways
- Model performance is only marginally better than baseline  
- Short-term Bitcoin price direction shows weak predictability  
- Reinforces market efficiency assumptions  

## 12. Conclusion
This project demonstrates an honest data analytics workflow focused on evaluation,
not hype or overfitting. The objective is insight, not trading advice.
